[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulnovello/Advanced-AI/blob/main/PP6%3A%SSL/ssl_part2_simclr.ipynb)
# SSL Part 2 — SimCLR: Contrastive Learning

## From Pretext Tasks to Contrastive Learning

In Notebook 1, we saw that **pretext tasks** (rotation, jigsaw, colorization) can learn useful features without labels. But they have key limitations:

- Each task must be **manually designed** — and the features learned are biased toward the specific task
- Rotation features are sensitive to orientation but may ignore texture
- Jigsaw features capture spatial structure but may miss color or fine detail

**Contrastive learning** takes a fundamentally different approach:

> Instead of solving a hand-crafted task, learn representations that are **invariant to data augmentations**.

The idea is simple:
1. Take an image, create **two augmented views** of it
2. Train the network to recognize that these two views came from **the same image**
3. At the same time, push apart views from **different images**

This forces the backbone to learn features that capture *what* the image contains (semantic content) while ignoring *how* it was augmented (appearance variations).

---

### What We Will Do

In this notebook, we will:

1. Build the **augmentation pipeline** — the most critical component of SimCLR
2. Implement the **SimCLR model**: backbone + projection head
3. Implement the **NT-Xent loss** (contrastive objective)
4. Train SimCLR and evaluate via **linear probing** and **t-SNE**
5. Investigate the effect of the **temperature** hyperparameter

---

### SimCLR at a Glance

| Component | Role |
|---|---|
| **Augmentation pipeline** | Creates two views of the same image (random crop, color jitter, flip, grayscale) |
| **Backbone** $f(\cdot)$ | Extracts representation $\mathbf{h} \in \mathbb{R}^{256}$ — this is what we keep |
| **Projection head** $g(\cdot)$ | Maps $\mathbf{h}$ to $\mathbf{z} \in \mathbb{R}^{128}$ — used only during training |
| **NT-Xent loss** | Pulls positive pairs together, pushes negatives apart in $\mathbf{z}$-space |

---
## Setup

In [ ]:
# !pip install -q torch torchvision matplotlib scikit-learn tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
from tqdm.notebook import tqdm as tqdm_nb
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

We will work with **CIFAR-10**, the same dataset as Notebook 1. This allows direct comparison with pretext task baselines.

In [ ]:
cifar10_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
cifar10_test  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

CIFAR10_CLASSES = cifar10_train.classes
print(f"Train: {len(cifar10_train)} images | Test: {len(cifar10_test)} images")
print(f"Classes: {CIFAR10_CLASSES}")

---
## Shared Backbone & Utilities

We reuse the **same backbone** and training conventions from Notebook 1. The key difference in SimCLR is that we will add a **projection head** on top of this backbone.

All utility functions below follow the same patterns as Notebook 1 so both notebooks stay consistent and results are directly comparable.

In [ ]:
# --- Shared backbone: same SmallConvNet as Notebook 1 ---

class SmallConvNet(nn.Module):
    """Small CNN backbone: 3 conv blocks -> adaptive pool -> 256-d feature vector."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(4),
        )
        self.fc = nn.Linear(128 * 4 * 4, 256)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


_test_backbone = SmallConvNet()
n_params = sum(p.numel() for p in _test_backbone.parameters())
print(f"Backbone parameters: {n_params:,}")
print(f"Output shape: {_test_backbone(torch.randn(1, 3, 32, 32)).shape}")

In [ ]:
# --- Shared constants and transform ---

NORMALIZE = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

In [ ]:
# --- Shared training loop (same as Notebook 1) ---

def train(model, train_loader, criterion, num_epochs=10, lr=1e-3,
          val_loader=None, task_type="classification", device=device):
    """Unified training loop for classification and regression tasks.

    Args:
        task_type: "classification" (tracks accuracy) or "regression" (tracks val loss).

    Returns:
        history dict with 'train_loss' and optionally 'val_acc' or 'val_loss'.
    """
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": []}
    if task_type == "classification":
        history["val_acc"] = []
    else:
        history["val_loss"] = []

    for epoch in tqdm_nb(range(1, num_epochs + 1), desc="Training", leave=False):
        model.train()
        total_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * inputs.size(0)
        avg_loss = total_loss / len(train_loader.dataset)
        history["train_loss"].append(avg_loss)

        if val_loader is not None:
            model.eval()
            with torch.no_grad():
                if task_type == "classification":
                    correct, total = 0, 0
                    for inputs, targets in val_loader:
                        inputs, targets = inputs.to(device), targets.to(device)
                        preds = model(inputs).argmax(dim=1)
                        correct += (preds == targets).sum().item()
                        total += targets.size(0)
                    val_metric = correct / total
                    history["val_acc"].append(val_metric)
                    print(f"  Epoch {epoch:2d}/{num_epochs} — loss: {avg_loss:.4f} — val acc: {val_metric:.1%}")
                else:
                    val_total = 0.0
                    for inputs, targets in val_loader:
                        inputs, targets = inputs.to(device), targets.to(device)
                        val_total += criterion(model(inputs), targets).item() * inputs.size(0)
                    val_metric = val_total / len(val_loader.dataset)
                    history["val_loss"].append(val_metric)
                    print(f"  Epoch {epoch:2d}/{num_epochs} — train loss: {avg_loss:.4f} — val loss: {val_metric:.4f}")
        else:
            print(f"  Epoch {epoch:2d}/{num_epochs} — loss: {avg_loss:.4f}")

    return history

In [ ]:
# --- Shared plotting ---

def plot_training_curves(history, task_name):
    """Plot training loss and validation metric (accuracy or loss)."""
    epochs = range(1, len(history["train_loss"]) + 1)

    has_val = "val_acc" in history or "val_loss" in history
    if has_val:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    else:
        fig, ax1 = plt.subplots(1, 1, figsize=(8, 4))

    ax1.plot(epochs, history["train_loss"], marker="o")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Train loss")
    ax1.set_title(f"{task_name} — Training loss")

    if "val_acc" in history:
        ax2.plot(epochs, [a * 100 for a in history["val_acc"]], marker="o", color="green")
        ax2.set_ylabel("Val accuracy (%)")
        ax2.set_title(f"{task_name} — Validation accuracy")
        ax2.set_xlabel("Epoch")
    elif "val_loss" in history:
        ax2.plot(epochs, history["val_loss"], marker="o", color="orange")
        ax2.set_ylabel("Val loss")
        ax2.set_title(f"{task_name} — Validation loss")
        ax2.set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Shared evaluation: feature extraction and t-SNE ---

@torch.no_grad()
def extract_features(backbone, dataset, device=device, batch_size=512):
    """Pass all images through a frozen backbone and return (features, labels) as numpy arrays."""
    backbone.eval()
    backbone.to(device)

    transform = T.Compose([T.ToTensor(), T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

    all_features, all_labels = [], []
    for i in range(0, len(dataset), batch_size):
        batch_imgs, batch_labels = [], []
        for j in range(i, min(i + batch_size, len(dataset))):
            img, label = dataset[j]
            batch_imgs.append(transform(img))
            batch_labels.append(label)

        imgs = torch.stack(batch_imgs).to(device)
        features = backbone(imgs).cpu().numpy()
        all_features.append(features)
        all_labels.extend(batch_labels)

    return np.concatenate(all_features, axis=0), np.array(all_labels)


def plot_tsne(test_feats, test_labels, title, n_samples=2000):
    """Run t-SNE on backbone features and plot colored scatter + image thumbnails."""
    indices = np.random.choice(len(test_feats), n_samples, replace=False)
    feats_subset = test_feats[indices]
    labels_subset = test_labels[indices]

    print(f"Running t-SNE on {n_samples} samples...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
    emb = tsne.fit_transform(feats_subset)
    print("Done!")

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    cmap = plt.cm.tab10
    for class_idx in range(10):
        mask = labels_subset == class_idx
        axes[0].scatter(emb[mask, 0], emb[mask, 1], c=[cmap(class_idx)],
                        label=CIFAR10_CLASSES[class_idx], s=8, alpha=0.6)
    axes[0].legend(fontsize=9, markerscale=3, loc="best")
    axes[0].set_title(f"t-SNE of {title} features (colored by class)", fontsize=13)
    axes[0].set_xlabel("t-SNE dim 1")
    axes[0].set_ylabel("t-SNE dim 2")

    N_THUMBNAILS = 200
    thumb_indices = np.random.choice(n_samples, N_THUMBNAILS, replace=False)
    axes[1].set_title(f"t-SNE with image thumbnails ({title})", fontsize=13)
    axes[1].set_xlabel("t-SNE dim 1")
    axes[1].set_ylabel("t-SNE dim 2")
    axes[1].scatter(emb[:, 0], emb[:, 1], c="lightgray", s=2, alpha=0.3)

    for ti in thumb_indices:
        global_idx = indices[ti]
        img_pil = cifar10_test[global_idx][0]
        img_np = np.array(img_pil)
        imagebox = OffsetImage(img_np, zoom=0.6)
        ab = AnnotationBbox(imagebox, (emb[ti, 0], emb[ti, 1]), frameon=False, pad=0)
        axes[1].add_artist(ab)

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Helper to build data loaders ---

def make_loaders(train_ds, val_ds, batch_size=256):
    """Create train and val DataLoaders with standard settings."""
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return train_loader, val_loader

---
## Section 1 — Augmentation Pipeline

SimCLR's core idea is deceptively simple: take one image, create **two different augmented views** of it, and train the network to recognize that these two views came from the same image — while pushing apart views from different images.

The augmentation pipeline is **the most critical component** of SimCLR. Chen et al. (2020) showed through extensive ablations that:

- **Random cropping + color jittering** is the most important combination
- Without strong augmentations, the model can "cheat" by relying on low-level shortcuts (e.g., matching color histograms or spatial position)
- Color distortion alone or cropping alone is not enough — it is their **composition** that forces the network to learn semantic features

### The augmentation recipe

| Transform | Purpose |
|---|---|
| `RandomResizedCrop(32, scale=(0.2, 1.0))` | Forces the network to recognize objects at different scales and positions |
| `RandomHorizontalFlip(p=0.5)` | Invariance to horizontal orientation |
| `ColorJitter(0.4, 0.4, 0.4, 0.1)` with $p=0.8$ | Prevents reliance on color shortcuts |
| `RandomGrayscale(p=0.2)` | Further discourages color-based shortcuts |

In [ ]:
# --- SimCLR augmentation pipeline ---

simclr_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # same normalization as NORMALIZE
])


class SimCLRDataset(Dataset):
    """Wraps a torchvision dataset to return two independently augmented views per image.

    Labels are completely ignored — contrastive learning is self-supervised.
    """

    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, _ = self.base_dataset[idx]  # label is ignored
        view1 = self.transform(img)
        view2 = self.transform(img)
        return view1, view2


simclr_dataset = SimCLRDataset(cifar10_train, simclr_transform)
print(f"SimCLR dataset: {len(simclr_dataset)} image pairs")

In [ ]:
# --- Visualize augmented pairs ---

def denormalize(tensor):
    """Undo Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) for display."""
    return tensor * 0.5 + 0.5

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    v1, v2 = simclr_dataset[i]
    axes[0, i].imshow(denormalize(v1).permute(1, 2, 0).clamp(0, 1))
    axes[0, i].set_title("View 1", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(denormalize(v2).permute(1, 2, 0).clamp(0, 1))
    axes[1, i].set_title("View 2", fontsize=9)
    axes[1, i].axis("off")
fig.suptitle("SimCLR augmented pairs — same image, different views", fontsize=13)
plt.tight_layout()
plt.show()

> **Question:** Why do we apply the transform **twice independently** rather than once? What would happen if both views received the same augmentation?
>
> *Hint:* If both views are identical, what trivial solution could the network learn? Think about what the contrastive loss would optimize for in that case.

---
## Section 2 — SimCLR Model

The SimCLR architecture has two components:

### 1. Backbone encoder $f(\cdot)$

The **exact same `SmallConvNet`** from Notebook 1. It produces a representation:

$$\mathbf{h} = f(\mathbf{x}) \in \mathbb{R}^{256}$$

This is the representation we **keep** for downstream tasks.

### 2. Projection head $g(\cdot)$

A 2-layer MLP that maps the backbone representation down to a lower-dimensional projection:

$$\mathbf{z} = g(\mathbf{h}) = W_2 \cdot \text{ReLU}(W_1 \cdot \mathbf{h}) \in \mathbb{R}^{128}$$

The projections are then **L2-normalized** so that cosine similarity reduces to a simple dot product: $\text{sim}(\mathbf{z}_i, \mathbf{z}_j) = \mathbf{z}_i^\top \mathbf{z}_j$.

### Why a separate projection head?

This is one of the key findings of the SimCLR paper. The contrastive loss **discards information** that is not useful for distinguishing augmented views — for example, color information may be removed since color jitter makes it unreliable.

By applying the contrastive loss to $\mathbf{z}$ rather than $\mathbf{h}$, the projection head acts as a **buffer** that absorbs this information loss. The backbone representation $\mathbf{h}$ remains richer and more general-purpose.

In [ ]:
# --- SimCLR model: backbone + projection head ---

class SimCLR(nn.Module):
    """SimCLR model: backbone encoder + 2-layer MLP projection head."""

    def __init__(self, backbone, feature_dim=256, projection_dim=128):
        super().__init__()
        self.backbone = backbone
        self.projection_head = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feature_dim, projection_dim),
        )

    def forward(self, x):
        h = self.backbone(x)                # representation (256-d) — kept for downstream
        z = self.projection_head(h)         # projection (128-d) — used only during training
        z = F.normalize(z, dim=1)           # L2 normalize
        return h, z


backbone = SmallConvNet()
model = SimCLR(backbone).to(device)
print(f"SimCLR total parameters: {sum(p.numel() for p in model.parameters()):,}")

> **Important:** After training, we **discard** the projection head and only keep the backbone. The 256-d representation $\mathbf{h}$ is what we evaluate via linear probing — exactly like we evaluated pretrained backbones in Notebook 1.
>
> This is analogous to how we discarded the rotation/jigsaw heads after pretraining — the head is a tool for learning, not for inference.

---
## Section 3 — NT-Xent Loss

The **NT-Xent** (Normalized Temperature-scaled Cross-Entropy) loss is the contrastive objective used in SimCLR.

### How it works

For a batch of $N$ images, we get $2N$ augmented views (two per image). For each view $i$, there is exactly **one positive** (its other augmented view $j$) and $2N - 2$ **negatives** (all other views in the batch).

$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(\mathbf{z}_i, \mathbf{z}_j) / \tau)}{\displaystyle\sum_{k \neq i} \exp(\text{sim}(\mathbf{z}_i, \mathbf{z}_k) / \tau)}$$

where:
- $\text{sim}(\mathbf{u}, \mathbf{v}) = \mathbf{u}^\top \mathbf{v} / (\|\mathbf{u}\| \|\mathbf{v}\|)$ is **cosine similarity**
- $\tau$ is the **temperature** parameter (controls sharpness of the distribution)

### Intuition

This is equivalent to a **$(2N-1)$-way classification problem**: for each view, pick the correct positive among $2N-1$ candidates. The temperature $\tau$ controls how much the model focuses on hard negatives vs. easy ones.

### Connection to InfoNCE

NT-Xent is a form of the **InfoNCE** loss (van den Oord et al., 2018), which is itself a lower bound on mutual information between the two views. Minimizing NT-Xent maximizes the mutual information between augmented views of the same image.

In [ ]:
# --- NT-Xent (Normalized Temperature-scaled Cross-Entropy) loss ---

class NTXentLoss(nn.Module):
    """NT-Xent contrastive loss for SimCLR."""

    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):
        """Compute NT-Xent loss.

        Args:
            z1: (N, proj_dim) L2-normalized projections from view 1
            z2: (N, proj_dim) L2-normalized projections from view 2

        Returns:
            Scalar loss.
        """
        N = z1.shape[0]

        # Concatenate all projections: (2N, proj_dim)
        z = torch.cat([z1, z2], dim=0)

        # Pairwise cosine similarity matrix: (2N, 2N)
        sim_matrix = torch.mm(z, z.T) / self.temperature

        # Mask out self-similarity (diagonal)
        mask = torch.eye(2 * N, dtype=torch.bool, device=z.device)
        sim_matrix = sim_matrix.masked_fill(mask, float("-inf"))

        # Positive pair labels:
        #   For i in [0..N-1], its positive is at index i+N
        #   For i in [N..2N-1], its positive is at index i-N
        labels = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)

        # Cross-entropy treats each row as logits, label = index of positive
        loss = F.cross_entropy(sim_matrix, labels)
        return loss

> **Question:** Can you see that NT-Xent is equivalent to a $(2N-1)$-way classification problem where the "correct class" for each view is its augmented counterpart?
>
> Look at the implementation: we build a $2N \times 2N$ similarity matrix, mask out the diagonal (self-similarity), and apply `cross_entropy` with the positive pair index as the label. This is exactly multi-class classification over $2N - 1$ candidates.

Let's verify with a quick unit test: with random embeddings, the loss should be close to $\log(2N - 1)$ (uniform guessing).

In [ ]:
# --- Unit test: NT-Xent with random embeddings ---

z1_test = F.normalize(torch.randn(4, 128), dim=1)
z2_test = F.normalize(torch.randn(4, 128), dim=1)
test_loss = NTXentLoss(temperature=0.5)(z1_test, z2_test)

expected = torch.log(torch.tensor(7.0))  # log(2N - 1) = log(7)
print(f"Loss with random embeddings: {test_loss.item():.4f}")
print(f"Expected ~log(2N-1) = log(7) = {expected.item():.4f}")

> With random (untrained) embeddings, all pairwise similarities are roughly equal, so the model is "guessing" uniformly among $2N-1$ candidates. The loss should be close to $\log(2N-1) = \log(7) \approx 1.946$.
>
> As training progresses, the loss should decrease well below this value — indicating that the model is successfully identifying positive pairs.

---
## Section 4 — Training SimCLR

SimCLR's contrastive training differs from the supervised loop in Notebook 1: each batch yields `(view1, view2)` pairs instead of `(input, label)`. We define a dedicated `train_simclr()` function.

### Training details

| Hyperparameter | Value | Rationale |
|---|---|---|
| Epochs | 10 | Enough to see clear improvement (original paper uses 1000) |
| Batch size | 256 | Larger is better for contrastive learning (more negatives per positive) |
| Learning rate | 3e-4 | Standard for Adam with this batch size |
| Weight decay | 1e-4 | Light regularization |
| Temperature $\tau$ | 0.3 | Controls sharpness of similarity distribution |

> **Note on batch size:** In contrastive learning, larger batches mean more negatives per positive pair, which improves the quality of the loss signal. The original SimCLR paper uses batch sizes of 4096–8192. Our batch size of 256 is small by comparison, but sufficient for demonstration.

In [ ]:
# --- SimCLR training function ---

SIMCLR_EPOCHS = 10
SIMCLR_LR = 3e-4
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.3


def train_simclr(model, train_loader, criterion, num_epochs=SIMCLR_EPOCHS,
                 lr=SIMCLR_LR, weight_decay=WEIGHT_DECAY, device=device):
    """Training loop for SimCLR contrastive pretraining.

    Follows the same conventions as Notebook 1's train() but handles
    the contrastive setup where each batch yields (view1, view2) pairs.

    Returns:
        history dict with 'train_loss'.
    """
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": []}

    for epoch in tqdm_nb(range(1, num_epochs + 1), desc="SimCLR Training", leave=False):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for view1, view2 in train_loader:
            view1, view2 = view1.to(device), view2.to(device)
            _, z1 = model(view1)
            _, z2 = model(view2)
            loss = criterion(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        avg_loss = total_loss / n_batches
        history["train_loss"].append(avg_loss)
        print(f"  Epoch {epoch:2d}/{num_epochs} — loss: {avg_loss:.4f}")

    return history

In [ ]:
# --- Train SimCLR ---

BATCH_SIZE = 256

train_loader = DataLoader(simclr_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)

criterion = NTXentLoss(temperature=TEMPERATURE)

print(f"Training: {SIMCLR_EPOCHS} epochs, batch size {BATCH_SIZE}, "
      f"lr={SIMCLR_LR}, tau={TEMPERATURE}")
print(f"Batches per epoch: {len(train_loader)}")

simclr_history = train_simclr(model, train_loader, criterion)
print("\nTraining complete!")

In [ ]:
plot_training_curves(simclr_history, "SimCLR")

> The **minimum possible loss is 0** (all positive pairs have similarity 1, all negatives have similarity 0), but in practice the loss stabilizes above 0 because with 256 pairs per batch there are $2 \times 256 - 2 = 510$ negatives competing for each positive — a challenging classification problem.
>
> If the loss stays close to $\log(511) \approx 6.24$, the model is not learning. A healthy training curve should show a clear downward trend.

---
## Section 5 — Linear Probing Evaluation

Now we answer the critical question: **are SimCLR's features better than pretext task features?**

We use the exact same evaluation protocol as Notebook 1:
1. **Freeze** the pretrained backbone (no gradient updates)
2. **Extract** 256-d features for all train and test images
3. **Fit** a logistic regression on top
4. **Report** test accuracy on CIFAR-10

This is a fair comparison because the backbone architecture, dataset, and evaluation method are identical across all experiments.

In [ ]:
# --- Linear probing: freeze backbone, extract features, fit logistic regression ---

def linear_probe(backbone, name, device=device):
    """Extract features from frozen backbone and fit a logistic regression on CIFAR-10.

    Features are pre-extracted with the backbone in eval mode to preserve
    BatchNorm running statistics from pretraining.

    Returns:
        (accuracy, test_feats, test_labels)
    """
    print(f"Extracting features from {name} backbone...")
    train_feats, train_labels = extract_features(backbone, cifar10_train, device)
    test_feats, test_labels   = extract_features(backbone, cifar10_test, device)

    print("Fitting linear probe...")
    clf = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", multi_class="multinomial")
    clf.fit(train_feats, train_labels)
    acc = clf.score(test_feats, test_labels)
    print(f"  → {name} linear probe accuracy: {acc:.1%}\n")

    return acc, test_feats, test_labels

In [ ]:
# --- Evaluate SimCLR backbone ---

print("=== SimCLR backbone ===")
simclr_probe_acc, simclr_test_feats, test_labels = linear_probe(backbone, "SimCLR")

print("\n=== Random baseline (untrained backbone) ===")
random_backbone = SmallConvNet()
random_probe_acc, random_test_feats, _ = linear_probe(random_backbone, "Random")

In [ ]:
# --- Comparison table ---

print("\n" + "=" * 55)
print(f"{'Method':<25} {'Linear Probe (CIFAR-10)':>25}")
print("-" * 55)
print(f"{'Random (baseline)':<25} {random_probe_acc:>24.1%}")
print(f"{'Rotation (NB1)':<25} {'~55%':>25}")
print(f"{'Jigsaw (NB1)':<25} {'~52%':>25}")
print(f"{'Colorization (NB1)':<25} {'~45%':>25}")
print(f"{'SimCLR (10 epochs)':<25} {simclr_probe_acc:>24.1%}")
print(f"{'Supervised (NB1)':<25} {'~80-85%':>25}")
print("=" * 55)
print("\nSimCLR significantly outperforms all pretext tasks from Notebook 1!")

> **Expected results:**
> - Random backbone: ~28%
> - SimCLR (10 epochs): ~65–70%
> - Rotation (Notebook 1): ~55%
> - Jigsaw (Notebook 1): ~52%
>
> Even with only 10 epochs on a tiny backbone, SimCLR already **surpasses all pretext tasks** from Notebook 1. The original paper trains for 1000 epochs on ImageNet with a ResNet-50 and reaches ~93% on CIFAR-10 — nearly matching supervised learning.

---
## Section 6 — Visualizing Learned Representations

Numbers tell part of the story, but **visualizations** help build intuition about what the model has actually learned. We use three complementary tools:

1. **t-SNE** — Projects 256-d features into 2D to reveal cluster structure
2. **Cosine similarity matrix** — Shows within-class vs. between-class similarity directly
3. **Positive/negative similarity histograms** — Shows how well the contrastive objective separated augmented pairs from different images

### 6.1 — t-SNE

We compare the feature space of a **random backbone** (no training) with the **SimCLR backbone** (contrastive pretraining). If SimCLR learned meaningful features, we should see tight, well-separated clusters by CIFAR-10 class — even though the model never saw class labels.

In [ ]:
# --- Extract features for visualization ---

simclr_test_feats, test_labels = extract_features(backbone, cifar10_test, device)
random_test_feats, _ = extract_features(random_backbone, cifar10_test, device)

print(f"Feature shape: {simclr_test_feats.shape}")

In [ ]:
plot_tsne(random_test_feats, test_labels, "random (untrained)")

In [ ]:
plot_tsne(simclr_test_feats, test_labels, "SimCLR-pretrained")

### What to observe

Even though the model **never saw a single class label** during training, the t-SNE reveals clear clustering by semantic class. Compare the two plots:

- **Random:** Points are scattered with no visible structure — the backbone hasn't learned anything useful
- **SimCLR:** Clear clusters emerge, with semantically similar classes (e.g., cat/dog, car/truck) sometimes overlapping — the contrastive objective has organized the feature space by semantic content

This is the core result: contrastive learning produces linearly separable features without any supervised signal.

### 6.2 — Cosine Similarity Matrix

Another way to visualize feature quality: compute pairwise **cosine similarity** between images from different classes.

We select 5 images per class (50 total), compute all $50 \times 50$ cosine similarities, and display them as a heatmap. A good representation should show:
- **High similarity** (warm colors) within each class block (diagonal blocks)
- **Low similarity** (cool colors) between different classes (off-diagonal blocks)

In [ ]:
# --- Cosine similarity matrix: 5 images per class ---

N_PER_CLASS = 5
selected_indices = []
for c in range(10):
    class_idx = np.where(test_labels == c)[0][:N_PER_CLASS]
    selected_indices.extend(class_idx)
selected_indices = np.array(selected_indices)

feats_sel = simclr_test_feats[selected_indices]
labels_sel = test_labels[selected_indices]

# Normalize and compute cosine similarity
feats_norm = feats_sel / (np.linalg.norm(feats_sel, axis=1, keepdims=True) + 1e-8)
sim_matrix = feats_norm @ feats_norm.T

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)

for i in range(1, 10):
    ax.axhline(i * N_PER_CLASS - 0.5, color="black", linewidth=0.5)
    ax.axvline(i * N_PER_CLASS - 0.5, color="black", linewidth=0.5)

tick_pos = [N_PER_CLASS * i + N_PER_CLASS // 2 for i in range(10)]
ax.set_xticks(tick_pos)
ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(tick_pos)
ax.set_yticklabels(CIFAR10_CLASSES, fontsize=8)
ax.set_title("Cosine similarity — SimCLR features (5 images/class)", fontsize=12)
plt.tight_layout()
plt.show()

The **block-diagonal pattern** confirms that images of the same class are more similar in the learned representation space. This structure emerged entirely from contrastive learning — no class labels were used.

Notice also that some off-diagonal blocks show higher similarity than others — this reflects genuine semantic relationships (e.g., cats and dogs are more similar to each other than to trucks).

### 6.3 — Positive vs Negative Similarity Distribution

Finally, let's directly verify what the NT-Xent loss optimizes: are **positive pairs** (two views of the same image) more similar than **negative pairs** (views of different images)?

We take a batch, compute projections $\mathbf{z}$, and plot the distributions of:
- **Positive similarities:** $\text{sim}(\mathbf{z}_i^{(1)}, \mathbf{z}_i^{(2)})$ — same image, different augmentations
- **Negative similarities:** $\text{sim}(\mathbf{z}_i^{(1)}, \mathbf{z}_j^{(2)})$ for $i \neq j$ — different images

In [ ]:
# --- Histogram: positive pair similarities vs negative pair similarities ---

model.eval()
sample_loader = DataLoader(simclr_dataset, batch_size=256, shuffle=True, drop_last=True)
view1, view2 = next(iter(sample_loader))

with torch.no_grad():
    _, z1 = model(view1.to(device))
    _, z2 = model(view2.to(device))

z1_np = z1.cpu().numpy()
z2_np = z2.cpu().numpy()

# Positive similarities: cosine(z1_i, z2_i)
pos_sims = np.sum(z1_np * z2_np, axis=1)

# Negative similarities: cosine(z1_i, z2_j) for i != j
all_sims = z1_np @ z2_np.T
neg_mask = ~np.eye(256, dtype=bool)
neg_sims = all_sims[neg_mask]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pos_sims, bins=50, alpha=0.7, color="#2e7d32", label="Positive pairs", density=True)
ax.hist(neg_sims, bins=50, alpha=0.7, color="#d44000", label="Negative pairs", density=True)
ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Density")
ax.set_title("Positive vs Negative Pair Similarity Distribution")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Positive similarity: mean={pos_sims.mean():.3f}, std={pos_sims.std():.3f}")
print(f"Negative similarity: mean={neg_sims.mean():.3f}, std={neg_sims.std():.3f}")

The clear **separation** between positive and negative distributions shows that SimCLR has successfully learned to:
- **Pull together** augmented views of the same image (high positive similarity)
- **Push apart** views from different images (low negative similarity)

This is exactly what the NT-Xent loss was designed to achieve. The degree of separation directly relates to the quality of learned features — more separation generally means better downstream performance.

---
## Section 7 — Bonus: Temperature Ablation

The temperature $\tau$ in the NT-Xent loss controls how "sharp" the similarity distribution is after softmax:

$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(\mathbf{z}_i, \mathbf{z}_j) / \tau)}{\sum_{k \neq i} \exp(\text{sim}(\mathbf{z}_i, \mathbf{z}_k) / \tau)}$$

| Temperature | Effect | Trade-off |
|---|---|---|
| **Low** $\tau$ (e.g., 0.1) | Sharp distribution — focuses on **hard negatives** | Can be unstable; gradients dominated by hardest examples |
| **Medium** $\tau$ (e.g., 0.3–0.5) | Balanced — the paper's sweet spot | Good gradient signal from a range of negatives |
| **High** $\tau$ (e.g., 1.0) | Smooth distribution — treats all negatives equally | Less discriminative; wastes capacity on easy negatives |

Let's train SimCLR at three different temperatures (5 epochs each) and compare linear probe accuracy.

In [ ]:
# --- Temperature ablation ---

ABLATION_EPOCHS = 5
temperatures = [0.1, 0.5, 1.0]
ablation_results = {}

for tau in temperatures:
    print(f"\n{'='*50}")
    print(f"Training with temperature tau={tau}")
    print(f"{'='*50}")

    abl_backbone = SmallConvNet()
    abl_model = SimCLR(abl_backbone).to(device)
    abl_criterion = NTXentLoss(temperature=tau)

    abl_history = train_simclr(abl_model, train_loader, abl_criterion,
                                num_epochs=ABLATION_EPOCHS)

    acc, _, _ = linear_probe(abl_backbone, f"tau={tau}")
    ablation_results[tau] = acc

In [ ]:
# --- Plot temperature ablation results ---

fig, ax = plt.subplots(figsize=(7, 4))
taus = list(ablation_results.keys())
accs = [ablation_results[t] * 100 for t in taus]
ax.bar([str(t) for t in taus], accs, color=["#d44000", "#2a5298", "#888888"])
ax.set_xlabel("Temperature (tau)")
ax.set_ylabel("Linear Probe Accuracy (%)")
ax.set_title(f"Temperature Ablation ({ABLATION_EPOCHS} epochs each)")
for i, (t, a) in enumerate(zip(taus, accs)):
    ax.text(i, a + 0.5, f"{a:.1f}%", ha="center", fontsize=11)
ax.set_ylim(0, max(accs) + 8)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

> **What to observe:**
>
> The temperature controls the trade-off between focusing on hard negatives (low $\tau$) and maintaining stable gradients (high $\tau$).
>
> - Too low ($\tau = 0.1$): training may be unstable, gradients dominated by a few hard negatives
> - Too high ($\tau = 1.0$): the loss treats all negatives equally, losing the ability to focus on informative comparisons
> - Sweet spot ($\tau \approx 0.3$–$0.5$): balanced gradient signal, best downstream performance
>
> In practice, $\tau$ is often treated as a fixed hyperparameter, though some methods (like CLIP) learn it during training.

---
## Summary

| Method | Linear Probe (CIFAR-10) | Source |
|---|---|---|
| Random baseline | ~28% | This notebook |
| Rotation | ~55% | Notebook 1 |
| Jigsaw | ~52% | Notebook 1 |
| Colorization | ~45% | Notebook 1 |
| **SimCLR (10 epochs)** | **~65–70%** | This notebook |
| Supervised (upper bound) | ~80–85% | Notebook 1 |

### Key Takeaways

1. **Augmentation is everything:** SimCLR's power comes from its augmentation pipeline (random crop + color jitter). Without strong augmentations, contrastive learning fails — the model finds trivial shortcuts instead of learning semantics.

2. **The NT-Xent loss** turns representation learning into a classification problem: "which of these $2N-2$ views is my augmented partner?" This is elegant because it requires no hand-designed pretext task.

3. **The projection head matters:** Training the contrastive loss on $\mathbf{z}$ (128-d) but evaluating on $\mathbf{h}$ (256-d) gives better downstream performance. The projection head absorbs information loss from the contrastive objective, keeping the backbone representation richer.

4. **SimCLR beats pretext tasks:** Even with 10 epochs, SimCLR significantly outperforms rotation, jigsaw, and colorization from Notebook 1. This is because contrastive learning captures general-purpose invariances rather than task-specific features.

5. **Temperature is a key hyperparameter:** It controls the trade-off between hard negative mining (low $\tau$) and training stability (high $\tau$). Values around 0.3–0.5 work best.

---

### What Comes Next?

SimCLR requires **negative pairs** to work — and performance improves with larger batch sizes (more negatives). This creates a practical limitation: you need large GPUs.

The next generation of SSL methods addresses this in different ways:
- **MoCo:** Uses a momentum-updated encoder and a queue to decouple batch size from the number of negatives
- **BYOL / SimSiam:** Eliminates negative pairs entirely using asymmetric architectures
- **Barlow Twins:** Replaces contrastive loss with a redundancy-reduction objective
- **MAE / DINOv2:** Uses masked image modeling instead of contrastive learning

We'll explore some of these in Notebook 3.